# 02. Tool-Calling Agent

**Topics covered:** Tool Calling · Function Calling · Memory

This notebook builds on [01_react_agent_from_scratch.ipynb](https://github.com/S33mi/modern-ai-llm-journey/blob/main/05_agents/01_react_agent_from_scratch.ipynb).

We will:
1. Represent tools as **JSON schemas** (function-calling style)
2. Build a **router + executor** that is reliable on small models / CPU
3. Add **conversation memory** (buffer of past turns)
4. Run multi-turn examples with tools + memory
5. Contrast this with free-form ReAct parsing

## 1. Setup

```bash
pip install transformers accelerate torch
```

In [62]:
# pip install transformers accelerate torch

In [63]:
import re
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

MODEL_NAME = "google/flan-t5-base" if DEVICE == "cuda" else "google/flan-t5-base" #google/flan-t5-small
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print(f"Model: {MODEL_NAME}")

Device: cpu


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model: google/flan-t5-base


In [64]:
def llm(prompt: str, max_new_tokens: int = 48) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

## 2. Tools as JSON Schemas (Function-Calling Style)

Modern APIs (OpenAI, Anthropic, Gemini, etc.) describe tools with a **name**, **description**, and **parameter schema**.  
The model returns a structured call; your code executes it.

We keep the same idea, but drive a small local model with multiple-choice + argument extraction (reliable on CPU).

In [65]:
TOOL_SPECS = [
    {
        "name": "calculator",
        "description": "Evaluate a math expression with + - * / and parentheses.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "e.g. 17*23 or (2+3)*4"}
            },
            "required": ["expression"],
        },
    },
    {
        "name": "convert",
        "description": "Convert units: km/miles, kg/lb, c/f.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "e.g. 10 km to miles"}
            },
            "required": ["query"],
        },
    },
    {
        "name": "search",
        "description": "Look up a short factual query in a tiny knowledge base.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "e.g. capital of japan"}
            },
            "required": ["query"],
        },
    },
]

def specs_as_text() -> str:
    lines = []
    for i, t in enumerate(TOOL_SPECS, 1):
        lines.append(f"{i}. {t['name']}: {t['description']}")
    lines.append(f"{len(TOOL_SPECS)+1}. none: answer without a tool")
    return "\n".join(lines)

print(specs_as_text())
print("\nExample schema:")
print(json.dumps(TOOL_SPECS[0], indent=2))

1. calculator: Evaluate a math expression with + - * / and parentheses.
2. convert: Convert units: km/miles, kg/lb, c/f.
3. search: Look up a short factual query in a tiny knowledge base.
4. none: answer without a tool

Example schema:
{
  "name": "calculator",
  "description": "Evaluate a math expression with + - * / and parentheses.",
  "parameters": {
    "type": "object",
    "properties": {
      "expression": {
        "type": "string",
        "description": "e.g. 17*23 or (2+3)*4"
      }
    },
    "required": [
      "expression"
    ]
  }
}


## 3. Tool Implementations (case-safe search)

In [66]:
def tool_calculator(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    expr = expression.strip()
    if not expr or not all(c in allowed for c in expr):
        return "Error: invalid expression"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


def tool_convert(query: str) -> str:
    q = query.lower().strip()
    m = re.match(r"([0-9.]+)\s*([a-z]+)\s+to\s+([a-z]+)", q)
    if not m:
        return "Error: use format like '10 km to miles'"
    value, src, dst = float(m.group(1)), m.group(2), m.group(3)
    table = {
        ("km", "miles"): value * 0.621371,
        ("miles", "km"): value * 1.60934,
        ("kg", "lb"): value * 2.20462,
        ("lb", "kg"): value * 0.453592,
        ("c", "f"): value * 9 / 5 + 32,
        ("f", "c"): (value - 32) * 5 / 9,
    }
    if (src, dst) not in table:
        return f"Error: unsupported {src} to {dst}"
    return f"{table[(src, dst)]:.4g} {dst}"


# Normalized lowercase KB keys
KB = {
    "capital of france": "Paris",
    "capital of japan": "Tokyo",
    "capital of germany": "Berlin",
    "capital of italy": "Rome",
    "capital of spain": "Madrid",
    "inventor of the telephone": "Alexander Graham Bell",
    "speed of light": "299792 km/s",
    "boiling point of water": "100 C",
}


def tool_search(query: str) -> str:
    q = query.lower().strip()
    q = re.sub(r"[^a-z0-9\s]", " ", q)
    q = re.sub(r"\s+", " ", q).strip()

    # exact / substring on normalized keys
    for k, v in KB.items():
        if k == q or k in q or q in k:
            return v

    # token overlap (order-insensitive)
    q_tokens = set(q.split())
    best_k, best_score = None, 0
    for k, v in KB.items():
        k_tokens = set(k.split())
        score = len(q_tokens & k_tokens)
        if score > best_score:
            best_score, best_k = score, k
    if best_k and best_score >= 2:
        return KB[best_k]

    return "No result found."


TOOL_IMPL = {
    "calculator": lambda args: tool_calculator(args.get("expression", "")),
    "convert": lambda args: tool_convert(args.get("query", "")),
    "search": lambda args: tool_search(args.get("query", "")),
}

# Quick search tests (case / wording)
for q in ["CAPITAL OF JAPAN", "What is the Capital of France?", "telephone inventor"]:
    print(f"{q!r:40s} → {tool_search(q)}")

'CAPITAL OF JAPAN'                       → Tokyo
'What is the Capital of France?'         → Paris
'telephone inventor'                     → Alexander Graham Bell


## 4. Router: which tool to call?

Small models: **multiple choice + keyword fallback** (same idea as the fixed ReAct notebook).

In [67]:
def route_tool(user_message: str, memory_text: str = "") -> str:
    """Return the best tool name or 'none'."""
    m = user_message.lower().strip()

    # -------------------------------------------------
    # 1. Strong keyword rules (most reliable)
    # -------------------------------------------------

    # Convert
    convert_patterns = [
        r"\bconvert\b", r"\bto miles\b", r"\bto km\b", r"\bto kilograms\b",
        r"\bto pounds\b", r"\bcelsius\b", r"\bfahrenheit\b",
        r"\d+\s*(km|miles|kg|lb|c|f|°c|°f)\b"
    ]
    if any(re.search(p, m) for p in convert_patterns):
        return "convert"

    # Calculator – only clear math
    has_digit = bool(re.search(r"\d", m))
    has_operator = bool(re.search(r"[\+\-\*/x×]|plus|minus|times|multiplied|divided", m))
    math_request = any(p in m for p in ["calculate", "compute", "what is", "how much is"])

    if has_digit and (has_operator or math_request):
        return "calculator"

    # Search
    search_keywords = [
        "capital", "who invented", "inventor", "who is", "what is the",
        "when was", "where is", "speed of light", "boiling point",
        "definition of", "meaning of", "who discovered"
    ]
    if any(k in m for k in search_keywords):
        return "search"

    # -------------------------------------------------
    # 2. LLM as last resort (only if keywords failed)
    # -------------------------------------------------
    prompt = f"""Choose the best tool. Reply with ONLY one word.

                Tools:
                - calculator
                - convert
                - search
                - none

                User message: {user_message}

                Answer:"""

    raw = llm(prompt, max_new_tokens=5).lower().strip()

    # Very strict matching
    if raw in ["calculator", "convert", "search", "none"]:
        return raw

    # If LLM said something messy, take the first valid word
    for name in ["calculator", "convert", "search", "none"]:
        if raw.startswith(name):
            return name

    return "none"


# ---------- Test ----------
test_messages = [
    "What is 9*8?",
    "10 km to miles",
    "capital of italy",
    "Hello there",
    "Convert 100 celsius to fahrenheit",
    "Who invented the telephone?",
    "Calculate 15 plus 27",
    "How are you today?",
    "What is the boiling point of water?",
    "25 * 4",
    "tell me a joke",
    "15 kg to lb"
]

print(f"{'Message':<45} → Tool")
print("-" * 55)
for msg in test_messages:
    print(f"{msg:<45} → {route_tool(msg)}")

Message                                       → Tool
-------------------------------------------------------
What is 9*8?                                  → calculator
10 km to miles                                → convert
capital of italy                              → search
Hello there                                   → none
Convert 100 celsius to fahrenheit             → convert
Who invented the telephone?                   → search
Calculate 15 plus 27                          → calculator
How are you today?                            → none
What is the boiling point of water?           → search
25 * 4                                        → calculator
tell me a joke                                → none
15 kg to lb                                   → convert


## 5. Argument filling (structured call object)

In [68]:
def fill_arguments(tool_name: str, user_message: str) -> dict:
    if tool_name == "calculator":
      prompt = f"""Extract ONLY the mathematical expression from the user message.
                Do not solve it. Do not add any extra words.

                Examples:
                User: What is 17*23?
                Expression: 17*23

                User: Calculate 12 + 8
                Expression: 12 + 8

                User: (5 + 3) * 2
                Expression: (5 + 3) * 2

                User: How much is 100 divided by 4?
                Expression: 100 / 4

                User: {user_message}
                Expression:"""

      expr = llm(prompt, max_new_tokens=20).strip()

      # Keep only valid characters
      expr = "".join(c for c in expr if c in "0123456789+-*/(). ")
      expr = expr.strip()

      # Extra cleaning: remove leftover words the model sometimes adds
      for bad in ["expression", "is", "calculate", "what", "result"]:
          expr = expr.replace(bad, "").strip()

      # Fallback: try to extract expression directly from user message
      if not expr or len(expr) < 2:
          match = re.search(r"[\d\.\s\+\-\*/\(\)]+", user_message)
          if match:
              expr = match.group(0).strip()

      return {"expression": expr or user_message}

    if tool_name == "convert":
        prompt = f"""Extract the conversion and write it in this EXACT format:

                      number from_unit to to_unit

                      Examples:
                      User: Convert 5 kg to lb
                      Output: 5 kg to lb

                      User: 10 kilometers to miles
                      Output: 10 km to miles

                      User: 100 celsius in fahrenheit
                      Output: 100 celsius to fahrenheit

                      User: {user_message}
                      Output:"""

        raw = llm(prompt, max_new_tokens=20).strip().lower()

        # Clean common LLM mistakes
        raw = raw.replace("convert ", "").replace("output:", "").replace("query:", "")
        raw = raw.split("\n")[0].strip()

        # Fallback with regex if LLM fails
        if not re.search(r"\d+\s*\w+\s+to\s+\w+", raw):
            match = re.search(
                r"(\d+\.?\d*)\s*(kg|lb|pounds?|km|kilometers?|miles?|celsius|fahrenheit|c|f)\s*(?:to|in|into)?\s*(kg|lb|pounds?|km|kilometers?|miles?|celsius|fahrenheit|c|f)?",
                user_message.lower()
            )
            if match:
                number, fr, to = match.groups()
                if to:
                    raw = f"{number} {fr} to {to}"
                else:
                    raw = f"{number} {fr}"

        return {"query": raw}

    if tool_name == "search":
        prompt = (
            "Extract a short lowercase search query form the given queury only.\n"
            "Examples: capital of japan | inventor of the telephone\n"
            f"User: {user_message}\nQuery:"
        )
        q = llm(prompt, max_new_tokens=24).strip().lower()
        q = re.sub(r"[^a-z0-9\s]", " ", q)
        q = re.sub(r"\s+", " ", q).strip()
        return {"query": q or user_message.lower()}

    return {}

def make_tool_call(user_message: str, memory_text: str = "") -> dict | None:
    """Return {name, arguments} or None if no tool needed."""
    name = route_tool(user_message, memory_text)
    if name == "none":
        return None
    args = fill_arguments(name, user_message)
    return {"name": name, "arguments": args}


print(make_tool_call("What is 17*23?"))
# → {'name': 'calculator', 'arguments': {'expression': '17*23'}}

print(make_tool_call("Convert 5 kg to lb"))
# → {'name': 'convert', 'arguments': {'query': '5 kg to lb'}}

print(make_tool_call("What is the CAPITAL of Itlay?")) #What is the CAPITAL of Japan?
# → {'name': 'search', 'arguments': {'query': 'capital of japan'}}

{'name': 'calculator', 'arguments': {'expression': '17*23'}}
{'name': 'convert', 'arguments': {'query': '5 kg to lb'}}
{'name': 'search', 'arguments': {'query': 'capital of itlay'}}


## 6. Conversation Memory

A simple **buffer memory** stores the last *N* turns so the agent can refer to prior tool results.

In [69]:
class BufferMemory:
    def __init__(self, max_turns: int = 6):
        self.max_turns = max_turns
        self.turns = []  # list of {role, content}

    def add(self, role: str, content: str):
        self.turns.append({"role": role, "content": content})
        if len(self.turns) > self.max_turns * 2:
            self.turns = self.turns[-self.max_turns * 2 :]

    def as_text(self) -> str:
        if not self.turns:
            return ""
        lines = []
        for t in self.turns:
            lines.append(f"{t['role'].upper()}: {t['content']}")
        return "\n".join(lines)

    def clear(self):
        self.turns.clear()


memory = BufferMemory()
print("Memory ready.")

Memory ready.


## 7. Agent Step: route → call → observe → reply

In [70]:
def agent_reply(user_message: str, memory: BufferMemory, verbose: bool = True) -> str:
    mem_text = memory.as_text()
    call = make_tool_call(user_message, mem_text)

    if call is None:
        # No tool – ask the LM to answer from memory / general knowledge (tiny model: keep short)
        prompt = (
            f"Ovel all Conversation:\n{mem_text}\n\n"
            f"User input: {user_message}\n"
            f"Assistant (brief):"
        )
        answer = llm(prompt, max_new_tokens=64)
        if verbose:
            print("Tool call: none")
    else:
        if verbose:
            print(f"Tool call: {json.dumps(call)}")
        result = TOOL_IMPL[call["name"]](call["arguments"])
        if verbose:
            print(f"Observation: {result}")

        # # Phrase a natural reply using the observation
        # prompt = (
        #     f"User asked: {user_message}\n"
        #     f"Tool {call['name']} returned: {result}\n"
        #     f"Write a short helpful answer for the user:"
        # )
        # answer = llm(prompt, max_new_tokens=64)

        # # If phrasing is empty/weird, fall back to raw result
        # if not answer or len(answer) < 2:
        #     answer = str(result)

        # ---------- Stronger reply generation ----------
        if call["name"] == "calculator":
            # Don't trust the small LLM to rephrase math — just return the result cleanly
            expression = call["arguments"].get("expression", "")
            answer = f"{expression} = {result}"

        elif call["name"] == "convert":
            answer = f"The result is {result}"

        else:
            # For search (and others) try to phrase it, but with a very strict prompt
            prompt = f"""Use the tool result to answer the user.
                        Do not invent any information.

                        User question: {user_message}
                        Tool result: {result}

                        Answer:"""
            answer = llm(prompt, max_new_tokens=50).strip()

            # Fallback if the model still hallucinates or returns empty
            if not answer or len(answer) < 3:
                answer = str(result)


    memory.add("user", user_message)
    memory.add("assistant", answer)
    return answer


print("Agent step ready.")

Agent step ready.


## 8. Single-turn demos

In [71]:
memory.clear()
for msg in [
    "What is 12 * 23?",            #"What is 17 * 23?"
    "Convert 10 km to miles",
    "What is the CAPITAL of Japan?",
    "Who invented the telephone?",
]:
    print("=" * 50)
    print("User:", msg)
    print("Agent:", agent_reply(msg, memory))
    print()

User: What is 12 * 23?
Tool call: {"name": "calculator", "arguments": {"expression": "12 * 23"}}
Observation: 276
Agent: 12 * 23 = 276

User: Convert 10 km to miles
Tool call: {"name": "convert", "arguments": {"query": "10 km to miles"}}
Observation: 6.214 miles
Agent: The result is 6.214 miles

User: What is the CAPITAL of Japan?
Tool call: {"name": "search", "arguments": {"query": "capital of japan"}}
Observation: Tokyo
Agent: Tokyo

User: Who invented the telephone?
Tool call: {"name": "search", "arguments": {"query": "inventor of the telephone"}}
Observation: Alexander Graham Bell
Agent: Alexander Graham Bell



## 9. Multi-turn with memory

The buffer lets the agent refer to earlier results.

In [72]:
memory.clear()

turns = [
    "Convert 100 c to f",
    "What was that result again?",          # relies on memory
    "Now search for the boiling point of water",
    "Thanks!",
]

for msg in turns:
    print("=" * 50)
    print("User:", msg)
    print("Agent:", agent_reply(msg, memory))
    print()

print("--- Memory dump ---")
print(memory.as_text())

User: Convert 100 c to f
Tool call: {"name": "convert", "arguments": {"query": "100 c to f"}}
Observation: 212 f
Agent: The result is 212 f

User: What was that result again?
Tool call: none
Agent: 212 f

User: Now search for the boiling point of water
Tool call: {"name": "search", "arguments": {"query": "search for the boiling point of water"}}
Observation: 100 C
Agent: 100 C

User: Thanks!
Tool call: {"name": "calculator", "arguments": {"expression": "1/100"}}
Observation: 0.01
Agent: 1/100 = 0.01

--- Memory dump ---
USER: Convert 100 c to f
ASSISTANT: The result is 212 f
USER: What was that result again?
ASSISTANT: 212 f
USER: Now search for the boiling point of water
ASSISTANT: 100 C
USER: Thanks!
ASSISTANT: 1/100 = 0.01


## 10. Function-calling vs ReAct (concept)

| Style | How the model acts | Pros | Cons |
|-------|--------------------|------|------|
| **ReAct text** | Emits Thought / Action prose | Interpretable traces | Fragile parsing on small models |
| **Function calling** | Emits structured `{name, arguments}` | Clean execution, API-native | Needs schema + reliable structured output |
| **This notebook** | Schema-defined tools + MC router | Works on FLAN-T5 CPU | Router is simpler than a full chat model |

Production stacks (OpenAI tools, Anthropic tools, Hugging Face `tool_call`, etc.) use the **schema + structured call** pattern. The agent loop is the same: decide → call → observe → answer → memory.

---

### Best practices learned:

- Give the model clear examples in the prompt
- Explicitly say “Do NOT answer the question” for search
- Always add cleaning + regex fallback
- Never fully trust the raw LLM output for structured arguments
---

### Biggest lesson:
Small models frequently ignore the tool result and hallucinate.
###Solution:

- For calculator and convert → Use a simple template instead of letting the LLM rephrase
- For search → Use a very strict prompt + fallback to the raw observation
- Prefer reliability over natural language when the model is weak

---

### Practical Rules for Building Reliable Tool Agents

- Never trust the LLM completely — always add rule-based fallbacks.
- Separate concerns clearly: Routing → Argument Extraction → Tool Execution → Answer Phrasing.
- Template the final answer for deterministic tools (calculator, convert).
- Test each stage independently before connecting them.
- Small models need much stronger prompts + cleaning than large models.
- Observation should be treated as ground truth — protect it from the LLM.

## 11. Summary

| Piece | Role |
|-------|------|
| **Tool schema** | Name, description, JSON parameters |
| **Router** | Pick tool (LM multiple-choice + keywords) |
| **Argument fill** | Build `{name, arguments}` |
| **Executor** | Run Python implementation |
| **Memory** | Buffer of recent user/assistant turns |
| **Reply** | Phrase observation for the user |

```python
call = make_tool_call(user_msg, memory.as_text())
if call:
    result = TOOL_IMPL[call["name"]](call["arguments"])
answer = phrase(user_msg, result)
memory.add("user", user_msg)
memory.add("assistant", answer)
```

---

**You have completed the `05_agents` [series](https://github.com/S33mi/modern-ai-llm-journey/blob/main/05_agents/).**

Next folder in the curriculum: **`06_projects`**  
→ [`01_rag_over_ml_notes.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/06_projects/)

---

**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Science/Analytics and ML/AI related opportunities